In [3]:
import pandas as pd
import os
from PIL import Image


# Load dataset
df = pd.read_csv('data.csv')

# Preview and filter
df['filepath'] = df['name'].apply(lambda x: os.path.join('Images', x))
train_df = df[df['is_training'] == 1]
test_df = df[df['is_training'] == 0]

# Check if image exists and is readable
def validate_image(path):
    try:
        img = Image.open(path)
        img.verify()
        return True
    except:
        return False

df['image_valid'] = df['filepath'].apply(validate_image)
print(f"Valid images: {df['image_valid'].sum()}/{len(df)}")

Valid images: 3962/4206


In [5]:
# Filter only valid images
df_clean = df[df['image_valid']]

# Keep only specified columns
df_clean = df_clean[['bmi', 'gender', 'is_training', 'name']]

# Save to CSV
df_clean.to_csv('clean_bmi_data.csv', index=False)

In [12]:
# I don't know if this part is useful.
# It uses MTCNN to detect and crop a face from an image.
# If it is not useful for the modeling. You can just delete it.


#!pip install opencv-python mtcnn
from mtcnn import MTCNN
import cv2

detector = MTCNN()

def extract_face(image_path, save_path, size=(224, 224)):
    img = cv2.imread(image_path)
    if img is None:
        return False
    results = detector.detect_faces(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if results:
        x, y, w, h = results[0]['box']
        face = img[y:y+h, x:x+w]
        face_resized = cv2.resize(face, size)
        cv2.imwrite(save_path, face_resized)
        return True
    return False